# Phase VII — fixed linear probe (full 60k)

This notebook reuses the frozen 60k feature matrix already stored in Drive, uses one common fixed LinearSVC(C=1) across all representations, checkpoints after every outer fold, and does not recompute image features or ordinal patterns.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, subprocess

REPO = Path('/content/painting-geometry')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch','multiscale-corpus-analysis','--single-branch',
                'https://github.com/ardominguezm/painting-geometry.git', str(REPO)], check=True)
commit = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
print('Repository commit:', commit)

ROOT = Path('/content/drive/MyDrive/painting_geometry_phase7_full')
INPUT = ROOT/'results'/'artbench_full_features_with_ordinal.csv'
OUTPUT = ROOT/'results'/'phase7_fixed_linear_probe'
LOG = ROOT/'phase7_fixed_linear_probe.log'
assert INPUT.exists(), f'Missing frozen feature matrix: {INPUT}'
print('Input :', INPUT, f'({INPUT.stat().st_size/1e6:.1f} MB)')
print('Output:', OUTPUT)
print('Log   :', LOG)


In [ ]:
import subprocess, sys

cmd = [sys.executable, '-u', str(REPO/'scripts'/'run_phase7_fixed_linear_probe.py'),
       '--features', str(INPUT), '--output-dir', str(OUTPUT),
       '--outer-folds', '5', '--n-boot', '5000']

with open(LOG, 'a', encoding='utf-8') as lf:
    lf.write(f'\n\n=== NEW FIXED-LINEAR RUN ===\ncommit={commit}\n')
    lf.flush()
    proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
        lf.write(line); lf.flush()
    rc = proc.wait()

if rc != 0:
    raise RuntimeError(f'Fixed linear probe stopped with exit code {rc}. Check {LOG}. Existing fold checkpoints are preserved.')
print('Fixed linear probe complete ✓')


In [ ]:
import zipfile

ZIP = ROOT/'painting_geometry_phase7_final_LIGHT.zip'
RESULTS = ROOT/'results'
include_dirs = [RESULTS/'phase7_full_style_geometry', RESULTS/'phase7_full_source_sensitivity', RESULTS/'phase7_fixed_linear_probe']
include_files = [RESULTS/'artbench_full_manifest.csv', RESULTS/'hf256_filename_recovery_audit.csv', RESULTS/'hf256_unlinked_rows.csv', RESULTS/'PHASE7_RUN_MANIFEST.json']
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for d in include_dirs:
        if d.exists():
            for p in d.rglob('*'):
                if not p.is_file(): continue
                if '_representation_checkpoints' in p.parts: continue
                z.write(p, p.relative_to(RESULTS))
    for p in include_files:
        if p.exists(): z.write(p, p.relative_to(RESULTS))
print('LIGHT ZIP ✓', ZIP, f'{ZIP.stat().st_size/1e6:.1f} MB')
